# BioSkills Lab — Chapter 7
## Biological Data as Matrices and PCA on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

In [ ]:
!pip install -q scanpy pandas numpy scikit-learn matplotlib umap-learn

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
print('Libraries loaded')

## Download TCGA Pan-Cancer Expression Data
We use a preprocessed subset from UCSC Xena.

In [ ]:
# Download TCGA Pan-Cancer expression data (preprocessed subset)
import urllib.request
url  = 'https://tcga.xenahubs.net/download/TCGA.PANCAN.sampleMap/HiSeqV2_PANCAN.gz'
path = 'tcga_expression.tsv.gz'
print('Downloading TCGA expression data...')
urllib.request.urlretrieve(url, path)
print('Done.')

In [ ]:
# Load and transpose
expr = pd.read_csv(path, sep='\t', index_col=0)
expr = expr.T
print(f'Samples: {expr.shape[0]}, Genes: {expr.shape[1]}')

In [ ]:
# Log2 transform
expr_log = np.log2(expr + 1)
print('Log2 transformed')

In [ ]:
# Load labels - TCGA cancer type annotations
labels_url = 'https://tcga.xenahubs.net/download/TCGA.PANCAN.sampleMap/PANCAN_clinicalMatrix'
urllib.request.urlretrieve(labels_url, 'tcga_labels.tsv')
labels = pd.read_csv('tcga_labels.tsv', sep='\t', index_col=0)
print(labels.columns[:10])

In [ ]:
# Align samples
common = expr_log.index.intersection(labels.index)
X = expr_log.loc[common].values
y = labels.loc[common, '_primary_disease'].values
print(f'Aligned samples: {len(common)}')

In [ ]:
# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f'Cancer types: {len(le.classes_)}')

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Scale - fit on train only
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# PCA - fit on train only
pca = PCA(n_components=50)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
print(f'PCA shape: {X_train_pca.shape}')

In [ ]:
# Variance explained
var = pca.explained_variance_ratio_
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.bar(range(1,51), var*100)
plt.xlabel('PC')
plt.ylabel('Variance (%)')
plt.title('Scree Plot')
plt.subplot(1,2,2)
plt.plot(np.cumsum(var)*100)
plt.xlabel('Number of PCs')
plt.ylabel('Cumulative variance (%)')
plt.title('Cumulative Variance')
plt.axhline(80, color='red', linestyle='--', label='80%')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Top 10 PCs explain: {sum(var[:10])*100:.1f}%')

In [ ]:
# PCA scatter coloured by cancer type
fig, ax = plt.subplots(figsize=(12,8))
for i, ct in enumerate(le.classes_):
    mask = y_enc == i
    ax.scatter(X_train_pca[y_train==i,0], X_train_pca[y_train==i,1],
               label=ct, alpha=0.4, s=8)
ax.set_xlabel(f'PC1 ({var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var[1]*100:.1f}%)')
ax.set_title('PCA of TCGA Pan-Cancer Expression')
ax.legend(bbox_to_anchor=(1.05,1), fontsize=6)
plt.tight_layout()
plt.show()